# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/h0pi/flyrank-ml-intenrship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is content-performance prioritisation. The task is a ranking problem: given a set of content pages, rank them by which pages the content team should review first.

The output is a priority score, not an automatic decision to change or remove a page. The top-ranked pages will be reviewed by a human who can decide whether to refresh content, investigate a technical issue, improve internal linking, or leave the page unchanged.

Ranking fits better than simple classification because the team has limited time and needs an ordered review queue, not only a yes/no label.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The starter-data proxy target is `is_declining_proxy`.

It is created from the observed `trend_direction` field:
- `1` when `trend_direction` is `down` (impressions in the last 30 days fell more than 20% compared with the 30 days before)
- `0` for every other value: `stable`, `up`, `flat` and `new`

This is a defined rule based on an observed directional trend, not a final business-outcome label. Two consequences:

- `trend_direction` and `trend_pct` are the label's source, so they can **never** be features. The same goes for `impressions_last_30d` / `impressions_prev_30d`, because `trend_pct` is computed from exactly those two columns.
- `new` pages (no impressions in the previous 30 days) end up as 0 even though nothing can be said about their trend. That is a limit of this proxy.

A stronger target, which I build from the warehouse in week 3, measures a real future window: features from before a decision date, the label from the days after it.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I will use Precision@50 as the primary success metric.

Precision@50 is the share of the 50 highest-priority pages that are actually declining according to the proxy target.

A starting target of Precision@50 >= 0.70 would mean that at least 35 of the first 50 pages reviewed are genuinely in the declining group. This is useful because the content team can act on a limited weekly review queue.

The number only means something next to the base rate: 54.2% of all pages are labelled declining, so picking 50 pages at random already gives about 0.54. A model has to beat that, and beat the transparent rule in section 5, before it adds value.

In [3]:
# Precision@50 target for the later ranking evaluation
k = 50
target_precision_at_k = 0.70

print(f"Primary metric: Precision@{k}")
print(f"Starting definition of good: >= {target_precision_at_k:.0%}")
print(f"That means at least {int(k * target_precision_at_k)} of the top {k} pages are declining.")

Primary metric: Precision@50
Starting definition of good: >= 70%
That means at least 35 of the top 50 pages are declining.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row represents one anonymised content page at the measurement snapshot used in the starter data.

Each row contains page-level signals that may help prioritise review, such as traffic or engagement measures, content attributes, and a directional trend field. The data is anonymised: no client names, URLs, or private search queries are used.

The dataframe below loads the content lane slice, creates the proxy target, and displays the available columns and a small sample of rows.

In [4]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

# Defined proxy target:
# 1 = the observed directional trend is down
# 0 = stable or up
df["is_declining_proxy"] = (
    df["trend_direction"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("down")
    .astype(int)
)

print(f"Loaded: {DATA_PATH}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("Unit of analysis: one row = one anonymised content page.")
print("\nTrend-direction distribution:")
display(
    df["trend_direction"]
    .value_counts(dropna=False)
    .rename_axis("trend_direction")
    .reset_index(name="page_count")
)

print("\nProxy-target distribution:")
display(
    df["is_declining_proxy"]
    .value_counts(dropna=False)
    .rename_axis("is_declining_proxy")
    .reset_index(name="page_count")
)

print("\nExample rows, including the proxy target:")
display(
    df[
        [
            "content_id",
            "content_type",
            "main_intent",
            "impressions_90d",
            "clicks_90d",
            "sessions_90d",
            "trend_direction",
            "trend_pct",
            "is_declining_proxy",
        ]
    ].head(10)
)

Loaded: data\raw\content_refresh_anonymized.csv
Rows: 30,000
Columns: 45
Unit of analysis: one row = one anonymised content page.

Trend-direction distribution:


,trend_direction,page_count
0,down,16262
1,stable,5962
2,up,4388
3,new,2236
4,flat,1152



Proxy-target distribution:


,is_declining_proxy,page_count
0,1,16262
1,0,13738



Example rows, including the proxy target:


,content_id,content_type,main_intent,impressions_90d,clicks_90d,sessions_90d,trend_direction,trend_pct,is_declining_proxy
0,content_304f48230142,keyword article,transactional,3803,29,17,down,-41.4,1
1,content_a1fb4e703a9e,keyword article,informational,15320,7,9,down,-57.7,1
2,content_9aa793d4d895,keyword article,informational,12581,11,11,down,-60.9,1
3,content_331d6c4de07b,keyword article,commercial,11751,58,78,stable,-13.8,0
4,content_d99b7a2d90ca,keyword article,informational,19140,24,145,down,-34.7,1
5,content_d4084a4bc775,keyword article,transactional,3970,1,5,down,-38.9,1
6,content_9a34b442b552,keyword article,informational,20,0,1,down,-92.3,1
7,content_a63219c6e95a,keyword article,commercial,1724,1,28,stable,0.6,0
8,content_5e6c160719bc,keyword article,informational,32574,29,68,down,-58.8,1
9,content_c27558df2b0c,keyword article,informational,1240,2,3,down,-29.2,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**A mistake I fixed first.** My first baseline ranked pages by `trend_direction == "down"` and then by `trend_pct`. It scored a perfect Precision@50 of 100%, but only because those two columns *are* the label. That is leakage, not a good rule. A baseline may only use signals that are known before the outcome.

**The honest baseline.** A rule a content team might really use: review the pages that have gone longest without an update first (`days_since_last_update`, ties broken by `content_age_days`). The cell below measures it. It lands at about the base rate, so staleness alone does not tell us which pages are declining.

That is why a single if-statement is not enough. Whether a page declines depends on several signals at once (visibility, position, CTR, content type, age, update history), and they interact: a stale page with strong, stable demand may be fine, while a recently updated page in a crowded topic can still slide. A model can learn how these signals combine into one consistent review order. It stays decision-support: a human reviews every recommendation before anything changes.

A future model must beat this rule (and the ~0.54 base rate) on Precision@50. If it doesn't, the simpler rule should be preferred.

In [5]:
# Honest transparent baseline: stalest pages first.
# Uses only signals known before the outcome — no trend_direction, no trend_pct,
# no last_30d / prev_30d columns (those are the label's source).

baseline_queue = (
    df.sort_values(
        by=["days_since_last_update", "content_age_days"],
        ascending=[False, False],
    )
    .head(50)
)

base_rate = df["is_declining_proxy"].mean()
baseline_precision_at_50 = baseline_queue["is_declining_proxy"].mean()

print("Baseline rule: review the pages with the most days since their last update first.")
print(f"Base rate (share of declining pages): {base_rate:.1%}")
print(f"Baseline Precision@50: {baseline_precision_at_50:.1%}")

display(
    baseline_queue[
        [
            "content_id",
            "days_since_last_update",
            "content_age_days",
            "is_declining_proxy",
        ]
    ].head(10)
)

Baseline rule: review the pages with the most days since their last update first.
Base rate (share of declining pages): 54.2%
Baseline Precision@50: 54.0%


,content_id,days_since_last_update,content_age_days,is_declining_proxy
4606,content_3f3576c295f5,373,374,0
26242,content_55a5b1c46474,373,374,1
29384,content_f6fdf87348f6,373,373,1
18440,content_8d56efff1e71,372,372,0
24216,content_1b4ec72dafd4,372,372,1
6962,content_f01216059a6a,335,335,1
8631,content_e2b702f4f92b,334,334,1
15608,content_06e19c6486b0,334,334,0
7509,content_7a888d3d99c8,313,313,1
15790,content_6476d1d8c050,313,313,0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.